In [2]:
from torch import optim
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
import torch
import random
import numpy as np
import torch.nn as nn
import albumentations as Albu
import pandas as pd
from torch.utils.data.sampler import RandomSampler
from warmup_scheduler import GradualWarmupScheduler
import os
from utils.dataset import PandasDataset
from utils.metrics import model_checkpoint
from utils.train import train_model
from utils.models import EfficientNetApi

In [3]:
seed = 42
shuffle = True
batch_size = 6
num_workers = 4
output_classes = 5
init_lr = 3e-3
warmup_factor = 2
warmup_epochs = 1
n_epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

ROOT_DIR = '../../..'

data_dir = '../../../../dataset'
images_dir = os.path.join(data_dir, 'tiles')

Using device: cuda


In [4]:
load_model = efficientnet_b0(
     weights=EfficientNet_B0_Weights.DEFAULT
)
model = EfficientNetApi(model=load_model, output_dimensions=output_classes, dropout_rate=0.6)
model = model.to(device)

In [5]:
print("Using device:", device)
loss_function = nn.BCEWithLogitsLoss()

torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

Using device: cuda


In [6]:
df_train_ = pd.read_csv(f"{ROOT_DIR}/data/train_5fold.csv")
df_train_.columns = df_train_.columns.str.strip()
train_indexes = np.where((df_train_['fold'] != 3))[0]
valid_indexes = np.where((df_train_['fold'] == 3))[0]
#
df_train = df_train_.loc[train_indexes]
df_val = df_train_.loc[valid_indexes]
df_test = pd.read_csv(f"{ROOT_DIR}/data/test.csv")

#### view data

In [7]:
(df_train.shape, df_val.shape, df_test.shape)

((7219, 5), (1805, 5), (1592, 4))

In [8]:
transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
])

In [9]:
df_train.columns = df_train.columns.str.strip()

train_dataset = PandasDataset(images_dir, df_train, transforms=transforms)
valid_dataset = PandasDataset(images_dir, df_val, transforms=None)
test_dataset = PandasDataset(images_dir, df_test, transforms=None)

In [10]:
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=batch_size, num_workers=num_workers, sampler=RandomSampler(train_dataset)
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(valid_dataset)
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=batch_size, num_workers=num_workers, sampler = RandomSampler(test_dataset)
)

In [11]:
optimizer = optim.Adam(model.parameters(), lr = init_lr / warmup_factor)
scheduler_cosine = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, n_epochs - warmup_epochs)
scheduler = GradualWarmupScheduler(optimizer, multiplier = warmup_factor, total_epoch = warmup_epochs, after_scheduler=scheduler_cosine)

In [12]:
print("\n=== B0 =========")
train_model(
    model=model,
    epochs=n_epochs,
    optimizer=optimizer,
    scheduler=scheduler,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    checkpoint=model_checkpoint,
    device=device,
    loss_function=loss_function,
    path_to_save_metrics="logs/b0-10-3.txt",
    path_to_save_model="models/b0.pth",
    patience=5,
)


=== B0 =========
Epoch 1/50



100%|██████████| 301/301 [01:35<00:00,  3.14it/s]


VAL_LOSS     0.410
VAL_ACC      Mean: 46.592 | Std: 1.150 | 95% CI: [44.654, 48.421]
VAL_KAPPA    Mean: 0.663 | Std: 0.015 | 95% CI: [0.637, 0.688]
VAL_F1       Mean: 0.348 | Std: 0.011 | 95% CI: [0.330, 0.366]
VAL_RECALL   Mean: 0.364 | Std: 0.010 | 95% CI: [0.348, 0.381]
VAL_PRECISION Mean: 0.471 | Std: 0.019 | 95% CI: [0.435, 0.501]
Salvando o melhor modelo... 0.0 -> 0.6626317780867319
Epoch 2/50



100%|██████████| 301/301 [01:37<00:00,  3.08it/s]
/home/woshington/Projects/Doutorado/repo/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:1087: UserWarning: To get the last learning rate computed by the scheduler, please use `get_last_lr()`.
  _warn_get_lr_called_within_step(self)


VAL_LOSS     0.352
VAL_ACC      Mean: 45.153 | Std: 1.183 | 95% CI: [43.213, 47.091]
VAL_KAPPA    Mean: 0.678 | Std: 0.014 | 95% CI: [0.655, 0.701]
VAL_F1       Mean: 0.360 | Std: 0.011 | 95% CI: [0.342, 0.379]
VAL_RECALL   Mean: 0.374 | Std: 0.011 | 95% CI: [0.357, 0.391]
VAL_PRECISION Mean: 0.483 | Std: 0.022 | 95% CI: [0.447, 0.517]
Salvando o melhor modelo... 0.6626317780867319 -> 0.6780858372071595
Epoch 3/50



loss: 0.56434, smooth loss: 0.32516:  88%|████████▊ | 1061/1204 [09:59<01:20,  1.77it/s]


KeyboardInterrupt: 

# tests

In [12]:
from utils.metrics import evaluation, format_metrics
model.load_state_dict(
    torch.load(f"models/b0.pth")
)
response = evaluation(model, test_loader, device)
result = format_metrics(response[0])
print(result)

  0%|          | 0/266 [00:01<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.27 GiB. GPU 0 has a total capacity of 11.76 GiB of which 1.03 GiB is free. Process 322099 has 8.62 GiB memory in use. Including non-PyTorch memory, this process has 750.00 MiB memory in use. Of the allocated memory 393.55 MiB is allocated by PyTorch, and 234.45 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
confusion_matrix = response[0].get("confusion_matrix")
print(confusion_matrix)